In [1]:
import pandas as pd
from cleaning_pipeline import clean_agent_data
t=pd.read_csv("agent_transactions.csv");a=pd.read_csv("agents.csv")
c,log=clean_agent_data(t,a)
n=pd.to_numeric(t.amount,errors="coerce")
audit=pd.DataFrame([
("Duplicate txn_id","Uniqueness","txn_id",t.txn_id.duplicated().sum()),
("Mixed case/spaces","Consistency","txn_type",(t.txn_type!=t.txn_type.str.strip().str.capitalize()).sum()),
("Missing or N/A","Completeness","amount",n.isna().sum()),
("Slash dates","Validity","txn_date",t.txn_date.str.contains("/").sum()),
("KES 250000","Accuracy","amount",(n==250000).sum()),
("Agent not in register","Timeliness","agent_id",(~t.agent_id.isin(a.agent_id)).sum())],
columns=["issue","dimension","column","rows"])
audit

,issue,dimension,column,rows
0,Duplicate txn_id,Uniqueness,txn_id,12
1,Mixed case/spaces,Consistency,txn_type,384
2,Missing or N/A,Completeness,amount,22
3,Slash dates,Validity,txn_date,5
4,KES 250000,Accuracy,amount,5
5,Agent not in register,Timeliness,agent_id,8


In [2]:
print(len(t),len(c),(c.county=="Unknown").sum());log

612 600 7


,rule,rows
0,duplicate txn_id removed,12
1,txn_type standardised,376
2,"amount missing/invalid (N/A, NaN)",20
3,"amount > IQR fence 6084 flagged, kept",40
4,txn_date with '/' fixed,5
5,"unknown agents (county=Unknown, kept)",7


In [3]:
c.to_csv("outputs/agent_transactions_clean.csv",index=False)
c.groupby(["county","txn_type"]).amount.agg(["sum","count","mean"])

sum  count         mean
county  txn_type                                
Kiambu  Deposit     334737.0     43  7784.581395
        Withdrawal   77024.5     27  2852.759259
Kisumu  Deposit     436501.5     74  5898.668919
        Withdrawal   76348.5     37  2063.472973
Mombasa Deposit     149094.0     72  2070.750000
        Withdrawal  368592.0     61  6042.491803
Nairobi Deposit     543698.0    118  4607.610169
        Withdrawal  161932.0     62  2611.806452
Nakuru  Deposit     184944.0     60  3082.400000
        Withdrawal  328568.5     39  8424.833333
Unknown Deposit      25279.0      6  4213.166667
        Withdrawal    1877.0      1  1877.000000

In [4]:
c.pivot_table(index=c.txn_date.dt.to_period("W"),columns="county",values="amount",aggfunc="sum")

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
txn_date,,,,,,
2026-02-23/2026-03-01,NaN,1893.0,6759.0,7046.0,2770.5,NaN
2026-03-02/2026-03-08,18317.0,287933.0,285696.5,68967.0,56574.0,NaN
2026-03-09/2026-03-15,267745.5,23186.0,30504.5,52887.5,28730.0,NaN
2026-03-16/2026-03-22,22054.0,28126.0,25171.5,43134.5,39795.0,1639.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41400.5,18972.5,NaN
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06/2026-04-12,15030.0,15341.0,47119.5,56813.5,276609.0,9824.0
2026-04-13/2026-04-19,12265.0,29470.0,26730.0,56406.5,26179.5,NaN
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65135.0,30862.0,1877.0


## Data Quality Report
- **Uniqueness:** 12 duplicate txn_ids removed (612 to 600).
- **Consistency:** txn_type unified to Deposit/Withdrawal.
- **Completeness:** 20 amounts missing/N/A; filled with type median, flagged amount_imputed.
- **Validity:** 5 dates used "/"; parsed to ISO.
- **Accuracy:** IQR flags 40 outliers (is_outlier); the 5 KES 250,000 rows are kept, flagged; verify with agents.
- **Timeliness:** 7 transactions use AG099, missing from the register; kept as county "Unknown"; update register.

**Privacy:** phone numbers, names and agent IDs linked to people are personal data. Under the Kenya Data Protection Act 2019, minimise, hash or mask phones, restrict access, and retain only as needed.